In [1]:
from pathlib import Path
import sys

import chromadb

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().resolve().parents] if (path / "pyproject.toml").exists()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(repo_root))

from app.config import CHROMA_HOST, CHROMA_PORT, CHROMA_SSL, COLLECTION_NAME
from app.factory import get_embeddings

client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT, ssl=CHROMA_SSL)
collection = client.get_collection(COLLECTION_NAME)
embeddings = get_embeddings()

/home/mahee/Work/Thesis/Repos/langchain-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Peek at stored documents
peek_results = collection.peek(5)
peek_results

{'ids': ['a72d43e4076871f5d8cb8e2b1c907243',
  'cc886c1201859e62060022f781b65db3',
  'a70b2f298c31b4f9fedde0e2acbe099f',
  '8538acc755194bec692214f1c6ed0299',
  'fb16a02def6666d74e43db85abc1dffa'],
 'embeddings': array([[-0.01735937,  0.01956889, -0.15263557, ..., -0.06103174,
         -0.05684036, -0.07121714],
        [-0.02163567,  0.08753343, -0.17609155, ..., -0.05169656,
         -0.02444785, -0.00216554],
        [-0.01206079,  0.10072881, -0.15852922, ..., -0.07825878,
         -0.01691264, -0.01559614],
        [ 0.03008747,  0.08734345, -0.16461468, ..., -0.05810713,
         -0.04446756, -0.02297426],
        [-0.0024467 ,  0.08375391, -0.1729205 , ..., -0.05865596,
         -0.04051889, -0.01436839]], shape=(5, 768)),
 'metadatas': [{'description': '',
   'source_file': 'knowledge_ingestion/content/v3/content/original paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.md',
   'source_corpus': 'unknown',
   'format': 'md',
  

In [ ]:
# Query exactly like your RAG retriever does — see what it returns
query = "What is the meaning of NISQ?"
query_embedding = embeddings.embed_query(query)
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {dist:.4f} | Source: {meta}")
    print(doc[:300])
    print("---")

Distance: 0.4922 | Source: {'section': '4.1 The 50-qubit barrier', 'content_type': 'narrative', 'description': '', 'title': 'Preskill - 2018 - Quantum Computing in the NISQ era and beyond', 'format': 'md', 'source_file': 'knowledge_ingestion/content/v3/content/original paper/Preskill - 2018 - Quantum Computing in the NISQ era and beyond.md', 'h2': '4.1 The 50-qubit barrier', 'source_corpus': 'unknown'}
## 4.1 The 50-qubit barrier  
Even with fault-tolerant quantum computing still a rather distant dream, we are now entering a pivotal new era in quantum technology. For this talk, I needed a name to describe this impending new era, so I made up a word: _NISQ_ . This stands for _Noisy IntermediateScal
---
Distance: 0.4969 | Source: {'content_type': 'narrative', 'source_file': 'knowledge_ingestion/content/v3/content/original paper/Preskill - 2018 - Quantum Computing in the NISQ era and beyond.md', 'source_corpus': 'unknown', 'format': 'md', 'section': 'John Preskill', 'title': 'Preskill - 2

In [ ]:
results

{'ids': [['d8f33811e8a1c1ab56c903be5c1d66ef',
   'dff57a6f7713e3776c0ac62e32f5db54',
   'd370f9fae61ba63427db4d35600588a6',
   '9df7908e6da955b03575de12165f2da6',
   '92d348a023f1b5db9a52cb7b3a8c5223']],
 'distances': [[0.527583, 0.6012751, 0.65522957, 0.6637537, 0.67982787]],
 'embeddings': None,
 'metadatas': [[{'source_file': 'knowledge_ingestion/content/v3/content/tech_docs/mlflow/docs/docs/classic-ml/deep-learning/transformers/tutorials/fine-tuning/transformers-peft.ipynb',
    'title': 'Fine-Tuning Open-Source LLM using QLoRA with MLflow and PEFT',
    'source_corpus': 'mlflow',
    'format': 'ipynb',
    'content_type': 'narrative',
    'section': '',
    'cell_index': 0},
   {'h1': 'Develop ML model with MLflow and deploy to Kubernetes',
    'description': '',
    'title': 'tutorial',
    'h2': 'Introduction: Scalable Model Serving with KServe and MLServer',
    'content_type': 'narrative',
    'source_file': 'knowledge_ingestion/content/v3/content/tech_docs/mlflow/docs/docs/cl